# 04 — Grad-CAM Explainability
**Plant Disease Detection (Residual CBM)** — Step 4 of 4

Self-contained notebook: no `scripts/` import. Uses real gradient-weighted
Grad-CAM — gradients of the predicted class w.r.t. the last conv feature map,
globally average-pooled into per-channel importance weights — not a plain
activation average (which is not class-discriminative).

**Prerequisites:** `best_model.keras` present.


In [ ]:
import os
import ast

import numpy as np
import pandas as pd
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt

IMG_SIZE = 224
SELECTED_CLASSES = [
    "Pepper__bell___Bacterial_spot", "Pepper__bell___healthy", "Potato___Early_blight",
    "Potato___Late_blight", "Potato___healthy", "Tomato_Bacterial_spot",
    "Tomato_Early_blight", "Tomato_Late_blight", "Tomato_Leaf_Mold",
    "Tomato_Septoria_leaf_spot", "Tomato_Spider_mites_Two_spotted_spider_mite",
    "Tomato__Target_Spot", "Tomato__Tomato_YellowLeaf__Curl_Virus",
    "Tomato__Tomato_mosaic_virus", "Tomato_healthy",
]

model_path = "best_model.keras" if os.path.exists("best_model.keras") else "../best_model.keras"
manifest_csv_path = "manifest.csv" if os.path.exists("manifest.csv") else "../manifest.csv"

model = tf.keras.models.load_model(model_path)
print(f"[INFO] Loaded {model_path} ({model.count_params():,} params)")

manifest = pd.read_csv(manifest_csv_path)
manifest["concept_vec"] = manifest["concept_vec"].apply(ast.literal_eval)
test_df = manifest[manifest["split"] == "test"]


## 4.1 Target conv layer
`activation_17` was verified once via a layer-shape inspection pass (see
`gradcam_inspect.txt` in the project root) — MobileNetV3 has many 4D-shaped
depthwise/expansion blocks, so this cannot be safely auto-detected at
inference time and is hardcoded after manual confirmation.

In [ ]:
def find_candidate_conv_layers(model):
    return [(layer.name, layer.output.shape) for layer in model.layers if len(layer.output.shape) == 4]

candidates = find_candidate_conv_layers(model)
cand_df = pd.DataFrame(candidates, columns=["layer_name", "output_shape"])
print(f"[INFO] {len(cand_df)} candidate 4D-output layers found. Last 8 (closest to the head):")
print(cand_df.tail(8).to_string(index=False))

LAST_CONV_LAYER_NAME = "activation_17"  # verified via gradcam_inspect.txt
assert LAST_CONV_LAYER_NAME in [c[0] for c in candidates], (
    f"'{LAST_CONV_LAYER_NAME}' not found in this model's layers -- re-inspect "
    f"the candidate list above before trusting the heatmaps below."
)
print(f"\n[OK] Using verified layer: '{LAST_CONV_LAYER_NAME}'")


## 4.2 Gradient-weighted Grad-CAM

In [ ]:
def make_gradcam_heatmap(model, img_batch, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output],
    )
    with tf.GradientTape() as tape:
        conv_output, predictions = grad_model(img_batch)
        _, disease_preds = predictions
        if pred_index is None:
            pred_index = int(tf.argmax(disease_preds[0]))
        class_channel = disease_preds[:, pred_index]

    grads = tape.gradient(class_channel, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_output = conv_output[0]
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), pred_index

def display_gradcam(img_path, true_class, ax_row):
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
    img_batch = np.expand_dims(img_resized, 0)

    heatmap, pred_idx = make_gradcam_heatmap(model, img_batch, LAST_CONV_LAYER_NAME)
    pred_class = SELECTED_CLASSES[pred_idx]

    heatmap_resized = cv2.resize(heatmap, (img_rgb.shape[1], img_rgb.shape[0]))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    superimposed = cv2.addWeighted(img_rgb, 0.6, heatmap_color, 0.4, 0)

    ax_row[0].imshow(img_rgb); ax_row[0].set_title(f"Original\ntrue: {true_class}", fontsize=9)
    ax_row[1].imshow(heatmap_resized, cmap="jet"); ax_row[1].set_title("Grad-CAM", fontsize=9)
    ax_row[2].imshow(superimposed); ax_row[2].set_title(f"Superimposed\npred: {pred_class}", fontsize=9)
    for ax in ax_row:
        ax.axis("off")

    correct = pred_class == true_class
    print(f"[RESULT] true='{true_class}'  pred='{pred_class}'  {'CORRECT' if correct else 'MISCLASSIFIED'}")
    return pred_idx

samples = test_df.sample(4, random_state=7)
fig, axes = plt.subplots(4, 3, figsize=(10, 13))
for row, (_, r) in zip(axes, samples.iterrows()):
    display_gradcam(r["File_Path"], r["Class_Name"], row)
plt.tight_layout()
plt.show()


## 4.3 Verify class-discriminativeness
Regenerates the heatmap for the same image against its top-2 predicted
classes and reports the numeric difference — mirrors what `tests/test_gradcam.py`
asserts automatically.

In [ ]:
row = test_df.sample(1, random_state=3).iloc[0]
img_bgr = cv2.imread(row["File_Path"])
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
img_batch = np.expand_dims(img_resized, 0)

_, disease_pred = model.predict(img_batch, verbose=0)
top2 = np.argsort(disease_pred[0])[::-1][:2]
print(f"[INFO] Image: {row['Class_Name']}")
print(f"[INFO] Top-2 predicted classes: {[SELECTED_CLASSES[i] for i in top2]}  "
      f"(probs: {[round(float(disease_pred[0][i]), 3) for i in top2]})")

heatmap_a, _ = make_gradcam_heatmap(model, img_batch, LAST_CONV_LAYER_NAME, pred_index=int(top2[0]))
heatmap_b, _ = make_gradcam_heatmap(model, img_batch, LAST_CONV_LAYER_NAME, pred_index=int(top2[1]))

mean_abs_diff = float(np.mean(np.abs(heatmap_a - heatmap_b)))
print(f"[RESULT] Mean absolute heatmap difference between top-2 classes: {mean_abs_diff:.4f}")
if mean_abs_diff > 0.01:
    print("[FINDING] Heatmaps differ meaningfully by target class -> confirms real,")
    print("[FINDING] class-discriminative Grad-CAM, not a plain activation average.")
else:
    print("[FINDING] Heatmaps are nearly identical across classes -> investigate before")
    print("[FINDING] trusting these visualizations as explanations.")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(heatmap_a, cmap="jet"); axes[0].set_title(SELECTED_CLASSES[top2[0]], fontsize=9); axes[0].axis("off")
axes[1].imshow(heatmap_b, cmap="jet"); axes[1].set_title(SELECTED_CLASSES[top2[1]], fontsize=9); axes[1].axis("off")
plt.suptitle("Same image, Grad-CAM for two different target classes")
plt.tight_layout()
plt.show()


## 4.4 Summary

Closes the 4-notebook pipeline. All four notebooks are now fully
self-contained — each can be opened and run independently regardless of the
kernel's working directory (no `scripts/` import, no `sys.path` dependency).

`scripts/*.py` remains the production source of truth used by `train.py`,
`quantize.py`, and the pytest suite — these notebooks are a documentation /
presentation layer that intentionally mirrors that logic rather than
replacing it.
